In [2]:
import arcpy
from pathlib import Path
import os
import shutil
import pandas as pd
from datetime import datetime


In [ ]:
# =============================================================================
# PATHS BASE - CL_MLP_PAO
# =============================================================================

# Carpeta de entrada donde se dejan los vuelos de drone sin procesar
PATH_INPUT_VUELOS_DRONE = r"\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_Drone_Sin_Procesar\INPUT"

# Excel base de parámetros del flujo
PATH_EXCEL_PARAMETROS = r"\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Parametros_ALL_v3.xlsx"

PATH_MOSAIC_DATASET = r"\\amssclgis08.ams.gmams.cl\CL_MLP_PAO\01_Proyectos_ArcGIS\APRX\CL MLP PAO Aereo Image Server_v2\SQLServer-amssclgis06_ArcGIS-Aereo.sde\OWD.C_MLP_PAO_IF_Ortho_Geosupport"


# =============================================================================
# PROYECTO ARCGIS PRO
# =============================================================================

# Proyecto APRX del Visor Territorial SIG PAO
PATH_APRX_VISOR_TERRITORIAL = r"\\amssclgis08.ams.gmams.cl\CL_MLP_PAO\01_Proyectos_ArcGIS\APRX\VISOR TERRITORIAL SIG PAO v7.aprx"


# =============================================================================
# GEODATABASES
# =============================================================================

# Geodatabase de imágenes PAO
PATH_GDB_IMAGENES = r"\\amssclgis08.ams.gmams.cl\CL_MLP_PAO\02_FGDB\CL_MLP_PAO_Imagenes.gdb"

# Geodatabase principal PAO v1
PATH_GDB_PAO_V1 = r"\\amssclgis08.ams.gmams.cl\CL_MLP_PAO\02_FGDB\CL_MLP_PAO_v1.gdb"


# =============================================================================
# FEATURE CLASSES - IMÁGENES
# =============================================================================

# Feature class de imágenes oblicuas de drone
PATH_FC_IMAGENES_OBLICUAS = (
    r"\\amssclgis08.ams.gmams.cl\CL_MLP_PAO\02_FGDB\CL_MLP_PAO_Imagenes.gdb"
    r"\CL_MLP_PAO_01_Imagenes_Oblicuas\CL_MLP_PAO_Oblicuas_Drone_v2"
)

# Índice / diccionario de vuelos PAO para imágenes
PATH_FC_INDICE_VUELOS_IMGS = (
    r"\\amssclgis08.ams.gmams.cl\CL_MLP_PAO\02_FGDB\CL_MLP_PAO_v1.gdb"
    r"\CL_MLP_PAO_06_COMPLEMENTOS\CL_MLP_PAO_Indice_Vuelos_PAO_IMGS_PO"
)


# =============================================================================
# FEATURE CLASSES - COMPLEMENTOS
# =============================================================================

# Feature class de macrozonas PAO
PATH_FC_MACROZONAS = (
    r"\\amssclgis08.ams.gmams.cl\CL_MLP_PAO\02_FGDB\CL_MLP_PAO_v1.gdb"
    r"\CL_MLP_PAO_06_COMPLEMENTOS\CL_MLP_PAO_ZONAS_Macrozonas_PO"
)


# =============================================================================
# FEATURE CLASSES - VIDEOS
# =============================================================================

# Feature class de videos de drone en terreno
PATH_FC_VIDEOS_DRONE = (
    r"\\amssclgis08.ams.gmams.cl\CL_MLP_PAO\02_FGDB\CL_MLP_PAO_v1.gdb"
    r"\CL_MLP_PAO_07_IMAGENES_TERRENO\MLP_SIG_PAO_Videos_Drone"
)


# =============================================================================
# CARPETAS DE VUELOS PROCESADOS / FECHA 26_06
# =============================================================================

# Carpeta de vuelos Drone - Chacay
PATH_DRONE_CHACAY_2606 = r"\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Chacay_Drone\26_06"

# Carpeta de vuelos Drone - Chacay El Mauro
PATH_DRONE_CHACAY_EL_MAURO_2606 = r"\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Chacay_El_Mauro_Drone\26_06"

# Carpeta de vuelos Drone - El Mauro
PATH_DRONE_EL_MAURO_2606 = r"\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\El_Mauro_Drone\26_06"

# Carpeta de vuelos Drone - El Mauro Puerto Punta Chungo
PATH_DRONE_EL_MAURO_PUERTO_PUNTA_CHUNGO_2606 = r"\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\El_Mauro_Puerto_Punta_Chungo_Drone\26_06"

# Carpeta de vuelos Drone - Puerto Punta Chungo
PATH_DRONE_PUERTO_PUNTA_CHUNGO_2606 = r"\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Puerto_Punta_Chungo_Drone\26_06"


# =============================================================================
# LISTAS AGRUPADAS
# =============================================================================

# Lista de carpetas de vuelos por zona
PATHS_CARPETAS_VUELOS_DRONE = [
    PATH_DRONE_CHACAY_2606,
    PATH_DRONE_CHACAY_EL_MAURO_2606,
    PATH_DRONE_EL_MAURO_2606,
    PATH_DRONE_EL_MAURO_PUERTO_PUNTA_CHUNGO_2606,
    PATH_DRONE_PUERTO_PUNTA_CHUNGO_2606,
]

# Diccionario general de paths principales
PATHS_PAO = {
    "input_vuelos_drone": PATH_INPUT_VUELOS_DRONE,
    "excel_parametros": PATH_EXCEL_PARAMETROS,
    "aprx_visor_territorial": PATH_APRX_VISOR_TERRITORIAL,
    "gdb_imagenes": PATH_GDB_IMAGENES,
    "gdb_pao_v1": PATH_GDB_PAO_V1,
    "fc_imagenes_oblicuas": PATH_FC_IMAGENES_OBLICUAS,
    "fc_indice_vuelos_imgs": PATH_FC_INDICE_VUELOS_IMGS,
    "fc_macrozonas": PATH_FC_MACROZONAS,
    "fc_videos_drone": PATH_FC_VIDEOS_DRONE,
    "carpetas_vuelos_drone": PATHS_CARPETAS_VUELOS_DRONE,
}

In [ ]:
# =============================================================================
# FASE 1 - CONFIGURACION DE CARGA DE IMAGENES
# =============================================================================

# Carpeta local del proyecto donde se dejan las imagenes nuevas a evaluar.
PATH_INPUT_LOCAL = Path.cwd() / "input"
PATH_INPUT_LOCAL.mkdir(parents=True, exist_ok=True)

IMAGE_EXTENSIONS = {
    ".tif",
    ".tiff",
    ".jpg",
    ".jpeg",
    ".png",
    ".sid",
    ".jp2",
    ".ecw",
}

PATH_INPUT_LOCAL

## Fase 1.1 - Buscar imagenes nuevas en `input/`

In [ ]:
def scan_input_images(input_folder, extensions=None):
    input_folder = Path(input_folder)
    extensions = {ext.lower() for ext in (extensions or IMAGE_EXTENSIONS)}

    if not input_folder.exists():
        raise FileNotFoundError(f"No existe la carpeta input: {input_folder}")

    rows = []

    for file_path in sorted(input_folder.rglob("*")):
        if not file_path.is_file() or file_path.suffix.lower() not in extensions:
            continue

        stat = file_path.stat()
        rows.append(
            {
                "file_name": file_path.name,
                "stem": file_path.stem,
                "extension": file_path.suffix.lower(),
                "path": str(file_path),
                "relative_path": str(file_path.relative_to(input_folder)),
                "size_mb": round(stat.st_size / (1024 * 1024), 3),
                "modified_at": datetime.fromtimestamp(stat.st_mtime),
            }
        )

    return pd.DataFrame(rows)


input_images_df = scan_input_images(PATH_INPUT_LOCAL)
print(f"Imagenes encontradas en input: {len(input_images_df)}")
input_images_df.head(20)

## Fase 1.2 - Revisar campos del mosaic dataset

In [ ]:
def list_dataset_fields(dataset_path):
    fields = arcpy.ListFields(dataset_path)
    return pd.DataFrame(
        [
            {
                "name": field.name,
                "alias": field.aliasName,
                "type": field.type,
                "length": field.length,
                "required": field.required,
                "editable": field.editable,
            }
            for field in fields
        ]
    )


mosaic_fields_df = list_dataset_fields(PATH_MOSAIC_DATASET)
print(f"Campos del mosaic dataset: {len(mosaic_fields_df)}")
mosaic_fields_df

## Fase 1.3 - Crear DataFrame del mosaic dataset

In [ ]:
def detect_candidate_path_fields(fields_df):
    tokens = ("path", "uri", "url", "file", "name", "source", "raster")
    candidate_fields = []

    for _, row in fields_df.iterrows():
        field_name = row["name"]
        field_type = row["type"]
        normalized_name = field_name.lower()

        if field_type in ("String", "Guid") and any(token in normalized_name for token in tokens):
            candidate_fields.append(field_name)

    return candidate_fields


def table_to_dataframe(dataset_path, fields=None, max_rows=5000):
    if fields is None:
        fields = [field.name for field in arcpy.ListFields(dataset_path) if field.type not in ("Geometry", "Raster", "Blob")]

    rows = []

    with arcpy.da.SearchCursor(dataset_path, fields) as cursor:
        for index, values in enumerate(cursor):
            if index >= max_rows:
                break

            rows.append(dict(zip(fields, values)))

    return pd.DataFrame(rows)


candidate_path_fields = detect_candidate_path_fields(mosaic_fields_df)
print("Campos candidatos para path/name del raster:", candidate_path_fields)

mosaic_df = table_to_dataframe(PATH_MOSAIC_DATASET, max_rows=5000)
print(f"Registros leidos del mosaic dataset: {len(mosaic_df)}")
mosaic_df.head(20)

## Fase 1.4 - Comparar `input/` contra el mosaic dataset

In [ ]:
def normalize_image_key(value):
    if value is None:
        return None

    value = str(value).strip().replace("/", "\\")

    if not value:
        return None

    return value.lower()


def build_mosaic_lookup(mosaic_df, candidate_fields):
    lookup = set()

    for field in candidate_fields:
        if field not in mosaic_df.columns:
            continue

        for value in mosaic_df[field].dropna():
            normalized_value = normalize_image_key(value)

            if normalized_value:
                lookup.add(normalized_value)
                lookup.add(Path(normalized_value).name.lower())
                lookup.add(Path(normalized_value).stem.lower())

    return lookup


mosaic_lookup = build_mosaic_lookup(mosaic_df, candidate_path_fields)

if input_images_df.empty:
    input_images_df = input_images_df.assign(exists_in_mosaic=[], match_key=[])
else:
    input_images_df = input_images_df.copy()
    input_images_df["match_key"] = input_images_df["path"].map(normalize_image_key)
    input_images_df["exists_in_mosaic"] = input_images_df.apply(
        lambda row: any(
            key in mosaic_lookup
            for key in (
                normalize_image_key(row["path"]),
                normalize_image_key(row["file_name"]),
                normalize_image_key(row["stem"]),
            )
            if key
        ),
        axis=1,
    )

new_images_df = input_images_df[input_images_df["exists_in_mosaic"] == False].copy()

print(f"Imagenes en input: {len(input_images_df)}")
print(f"Imagenes ya detectadas en mosaic dataset: {int(input_images_df['exists_in_mosaic'].sum()) if not input_images_df.empty else 0}")
print(f"Imagenes nuevas candidatas a cargar: {len(new_images_df)}")

new_images_df